In [ ]:
# Install dependencies (run once per fresh environment, before any imports;
# restart the kernel afterwards if packages were upgraded)
%pip install -q -r ../requirements.txt

In [ ]:
# --- Setup (Step 0) ---
import sys
from pathlib import Path

# Make src/ importable whether run locally (cwd=src/) or from repo root
for p in (Path.cwd(), Path.cwd() / "src"):
    if (p / "config.py").is_file() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import config as C
import text2sql_utils as U  # the same prompt, generation, extraction and scoring code as Task 1

C.set_seed()
C.get_hf_token()  # Gemma 3 is gated: raises with setup instructions if no HF login/token is found
FT = C.FINETUNE
print("repo root:", C.REPO_ROOT)
print("fine-tuning settings:", FT)
print("runtime:", C.runtime_info())

# Task 3 – RQ3: To what extent does fine-tuning on Spider's training data improve Gemma 3 1B?

In Task 1, Gemma 3 1B was the weakest model: it answered only about 30% of the dev questions correctly (EX), and about a quarter of its queries did not even run. Here we **fine-tune** it: we continue training the model on example questions and their correct SQL queries from Spider's **training** split, so that it learns to write SQL for a database schema like ours. Then we evaluate it **exactly like in Task 1**: the same 1034 dev questions, prompt, decoding settings, SQL extraction and official scoring. That way the difference between the original and the fine-tuned model comes from the training alone.

**Choices (made with the user):**
- **Full fine-tuning:** all of the model's ~1 billion weights are updated (rather than only small add-on weights, as LoRA would do).
- **Training data:** `train_spider.json` only (7000 examples over 140 databases). The extra `train_others.json` file (older, single-domain datasets) is not used.
- **Validation split by database:** about 10% of the training examples, held out by whole database, are used only to decide how long to train (which epoch to keep). The dev set is never used for training or for any choice; it is touched once, for the final evaluation.
- **One seed for everything:** `C.SEED = 42` (split, data order, initialization). Every split and setting is saved to files so it can be checked and reproduced.

All Task 1 code that is reused here (schema rendering, prompts, generation, SQL extraction, scoring) comes from `src/text2sql_utils.py`, the same functions Task 1 uses. Outputs: `outputs/task3/` (split, metrics, figures) and `outputs/gemma3-1b/<run>/` (generated SQL). The fine-tuned weights are saved in `checkpoints/`, which is never committed.

## Step 1.1 – Load the training data and build the prompts

This cell loads `train_spider.json` (7000 question–SQL pairs over 140 databases) and checks that none of these databases is a dev database. Spider's training and dev splits use different databases, so the final evaluation tests the model on databases it never saw.

Each training example gets **exactly the same prompt as in Task 1**: the template from `src/prompt_template.txt`, filled with the database schema (CREATE TABLE statements + 3 sample rows per table) and the question. Each example also gets its **official difficulty level**, computed the same way as for dev, so that the validation split can be compared with the training split.

The **target** the model learns to produce is the gold SQL query, with spaces and line breaks collapsed and any trailing semicolon removed, and nothing else: no explanation and no Markdown code block. That is the answer format the prompt asks for.

In [ ]:
import json
import statistics
from collections import Counter

U.setup_eval_script()  # official Spider evaluation code (needed for difficulty and scoring)
tables = U.load_tables()
train_all = U.load_spider(FT.train_file)  # train_spider.json
dev_inputs = json.loads(C.DEV_INPUTS.read_text())  # the Task 1 dev inputs (prompt, difficulty, gold SQL)
dev_dbs = {ex["db_id"] for ex in dev_inputs}

train_dbs = sorted({ex["db_id"] for ex in train_all})
assert not set(train_dbs) & dev_dbs, "a training database is also a dev database"
print(f"{len(train_all)} training examples over {len(train_dbs)} databases; no overlap with the {len(dev_dbs)} dev databases")

template = C.PROMPT_TEMPLATE.read_text()
schemas = {db: U.render_schema(tables[db], U.db_path(db)) for db in train_dbs}  # same format as Task 1


def target_sql(query: str) -> str:
    """The answer the model learns to write: the gold query on one line, without a trailing semicolon."""
    return " ".join(query.split()).rstrip(";").rstrip()


train_examples = []
for i, ex in enumerate(train_all):  # id = position in train_spider.json
    try:
        level = U.difficulty(ex["db_id"], ex["query"])
    except Exception:  # the official parser cannot read a few gold queries
        level = "unparsed"
    train_examples.append({"id": i, "db_id": ex["db_id"], "question": ex["question"], "query": ex["query"],
                           "target": target_sql(ex["query"]), "difficulty": level,
                           "prompt": U.build_prompt(template, ex["db_id"], schemas[ex["db_id"]], ex["question"])})

lengths = [len(ex["prompt"]) for ex in train_examples]
print(f"prompt length (chars): median {statistics.median(lengths):.0f}, max {max(lengths)}")
print("difficulty:", dict(Counter(ex["difficulty"] for ex in train_examples)))
print("example target:", train_examples[0]["question"], "->", train_examples[0]["target"])

## Step 1.2 – Validation split (by database, seed 42)

To decide **how long to train**, we need to measure the model on questions it was not trained on. The dev set must not be used for that, because it is reserved for the final evaluation. So we hold out part of the *training* split as a **validation set**.

**How the split is made:**
- **By whole database, not by single question.** All questions about a database go either to training or to validation. The dev set uses databases the model never saw, so the validation set should too; otherwise the model could do well just because it already knows the schema.
- **Seeded random order.** The 140 training databases are sorted by name, then shuffled with a random generator seeded with `C.SEED = 42`. Databases are taken in that order until their questions reach 10% of the training examples (`FT.val_fraction`).
- **Difficulty check.** We compare the difficulty mix (easy / medium / hard / extra) of the validation and training parts. A difference of more than 5 percentage points in any level is reported. We do **not** re-shuffle with another seed to make it look better, because that would make the seed choice arbitrary.

The split (seed, database lists, example ids and difficulty counts) is saved to `outputs/task3/splits.json`, so anyone can check exactly which examples were used where.

In [ ]:
import random

LEVELS = ["easy", "medium", "hard", "extra"]

# 1) shuffle the databases with the fixed seed (sorting first makes the result independent of file order)
rng = random.Random(FT.seed)
db_order = sorted(train_dbs)
rng.shuffle(db_order)

# 2) take whole databases until 10% of the examples are in the validation set
per_db = Counter(ex["db_id"] for ex in train_examples)
target_n = round(FT.val_fraction * len(train_examples))
val_dbs, n_val = [], 0
for db in db_order:
    if n_val >= target_n:
        break
    val_dbs.append(db)
    n_val += per_db[db]
fit_dbs = [db for db in db_order if db not in val_dbs]

train_ids = [ex["id"] for ex in train_examples if ex["db_id"] in fit_dbs]
val_ids = [ex["id"] for ex in train_examples if ex["db_id"] in val_dbs]

# 3) checks: disjoint, complete, no dev database
assert not set(train_ids) & set(val_ids) and len(train_ids) + len(val_ids) == len(train_examples)
assert not (set(fit_dbs) | set(val_dbs)) & dev_dbs


def level_shares(ids):
    c = Counter(train_examples[i]["difficulty"] for i in ids)
    return {lvl: round(c[lvl] / len(ids), 3) for lvl in LEVELS + ["unparsed"] if c[lvl]}


share_train, share_val = level_shares(train_ids), level_shares(val_ids)
max_gap = max(abs(share_train.get(l, 0) - share_val.get(l, 0)) for l in LEVELS)
print(f"training part: {len(train_ids)} examples over {len(fit_dbs)} databases; difficulty shares {share_train}")
print(f"validation part: {len(val_ids)} examples over {len(val_dbs)} databases; difficulty shares {share_val}")
print(f"largest difference in a difficulty share: {100 * max_gap:.1f} percentage points"
      + ("  -> more than 5 points: reported, not re-shuffled" if max_gap > 0.05 else "  (within 5 points)"))

splits = {"seed": FT.seed, "source": f"{FT.train_file}.json", "val_fraction": FT.val_fraction,
          "method": "whole databases, sorted by name then shuffled with random.Random(seed); taken until val_fraction of examples",
          "train_dbs": sorted(fit_dbs), "val_dbs": sorted(val_dbs), "train_ids": train_ids, "val_ids": val_ids,
          "train_difficulty": dict(Counter(train_examples[i]["difficulty"] for i in train_ids)),
          "val_difficulty": dict(Counter(train_examples[i]["difficulty"] for i in val_ids)),
          "dev": "all 1034 dev examples (src/spider_dev_inputs.json), used only for the final evaluation"}
C.TASK3_SPLITS.parent.mkdir(parents=True, exist_ok=True)
C.TASK3_SPLITS.write_text(json.dumps(splits, indent=1) + "\n")
print("saved", C.TASK3_SPLITS)

## Step 1.3 – Turn the examples into training sequences

A language model learns by predicting the next token of a text. Each training example becomes one text in Gemma 3's chat format: the **prompt** as the user's message, then the **target SQL** as the model's reply, ending with the end-of-turn token. That token teaches the model to stop after the query.

**Only the reply is learned.** The training signal (the "loss") is computed on the target SQL tokens only. The prompt tokens are marked to be ignored (label −100). The model should learn to *answer*, not to reproduce the long schema text in the prompt. This is a common choice for instruction fine-tuning, and it also keeps training fast.

This cell tokenizes all examples and checks that the reply ends with the end-of-turn token the model uses to stop at inference time.

In [ ]:
import transformers

spec = C.MODELS[FT.model_key]
tokenizer = transformers.AutoTokenizer.from_pretrained(spec.hf_id, revision=spec.revision, token=C.get_hf_token())
END_OF_TURN = "<end_of_turn>"
eot_id = tokenizer.convert_tokens_to_ids(END_OF_TURN)
assert eot_id in (tokenizer.eos_token_id if isinstance(tokenizer.eos_token_id, list) else [tokenizer.eos_token_id, 106])


def to_sequence(ex):
    """input_ids = chat-formatted prompt + target + end-of-turn; labels = -100 on the prompt (not learned)."""
    prompt_ids = tokenizer(U.build_input(spec, tokenizer, ex["prompt"]), add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(ex["target"] + END_OF_TURN, add_special_tokens=False)["input_ids"]
    assert target_ids[-1] == eot_id
    return {"input_ids": prompt_ids + target_ids, "labels": [-100] * len(prompt_ids) + target_ids}


sequences = {ex["id"]: to_sequence(ex) for ex in train_examples}
seq_len = [len(sequences[i]["input_ids"]) for i in train_ids]
tgt_len = [sum(t != -100 for t in sequences[i]["labels"]) for i in train_ids]
print(f"training sequences: {len(train_ids)}; length median {statistics.median(seq_len):.0f} tokens, max {max(seq_len)}; "
      f"target (learned) tokens median {statistics.median(tgt_len):.0f}, total {sum(tgt_len):,}")
ex0 = sequences[train_ids[0]]
print("end of the first sequence (decoded):", repr(tokenizer.decode(ex0["input_ids"][-40:])))

## Step 2.1 – Fine-tune the model (with a validation check after every epoch)

**What happens in training.** The model reads a batch of training sequences and predicts each next token of the target SQL. The **loss** measures how wrong those predictions were. An optimizer (AdamW) then adjusts all ~1 billion weights a little in the direction that lowers the loss. One pass over all 6282 training examples is an **epoch**; we train for up to 3 epochs.

**Settings** (`C.FINETUNE`; standard values for fine-tuning a model of this size, not tuned on dev):
- **Learning rate 1e-5:** how big each adjustment is. A small value changes the pre-trained model gently.
- **Batch of 32 examples per adjustment:** 8 examples are processed at a time and their results are added up over 4 passes before the weights are updated ("gradient accumulation"). This gives the effect of a larger batch on limited memory.
- **Warm-up and cosine schedule:** the learning rate rises from 0 over the first 3% of updates, then slowly decreases to 0, a common way to keep training stable.
- **Gradient clipping at 1.0:** caps unusually large updates.
- **Precision:** the weights are kept in 32-bit precision while the computations run in 16-bit (bf16) "mixed precision". With weights stored only in bf16, updates this small can be rounded away. Saved checkpoints are converted to bf16, the same precision as the Task 1 model.
- **Memory-saving steps** (these change nothing in the result):
  - the vocabulary scores are computed only at the target positions, because Gemma's vocabulary has 262k tokens;
  - gradient checkpointing recomputes intermediate results instead of storing them;
  - batches with very long sequences are processed in smaller pieces, with the loss weighted so the result is the same.
- **Batching:** examples are shuffled with a seed per epoch (`C.SEED + epoch`) and grouped with others of similar length to avoid wasted padding.

**Choosing the number of epochs.** Training too short leaves the model under-trained; training too long can make it memorize the training databases ("overfitting"). After every epoch we save the model and measure its **execution accuracy (EX) on the validation databases**, using the same generation and scoring code as Task 1. We also measure the original model on the validation set ("epoch 0") as a reference. The epoch with the highest validation EX is kept as the final model (if two tie, the earlier one). The dev set is not involved in this choice.

**Re-running:** training takes about an hour on one A40. If a finished model is already in `checkpoints/`, this cell skips training and reuses it. The training log and validation results are saved in `outputs/task3/metrics/`. GPU training is not bit-for-bit deterministic, so re-training can give slightly different weights; the evaluation of a saved model is deterministic.

In [ ]:
import gc
import math
import shutil
import time

import pandas as pd
import torch
import torch.nn.functional as F

RUN_DIR = C.CHECKPOINTS_DIR / "gemma3-1b-spider-ft"  # gitignored: weights are never committed
BEST_DIR = RUN_DIR / "best"
C.TASK3_METRICS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_FILE = C.TASK3_METRICS_DIR / "task3_training_summary.json"
PROGRESS_FILE = RUN_DIR / "progress.json"  # live progress, read by archive/progress.py (status line)
val_examples = {i: train_examples[i] for i in val_ids}
val_prompts = {i: train_examples[i]["prompt"] for i in val_ids}


def validation_ex(run_name, model_path):
    """Generate SQL for the validation questions (same code as Task 1) and score it with the official scorer."""
    stale = C.run_dir(FT.model_key, run_name) / "responses.jsonl"
    if model_path is not None and stale.exists():
        stale.unlink()  # outputs of an earlier training run: generate again for this checkpoint
    U.run_model(FT.model_key, val_ids, run_name, val_prompts, model_path=model_path)
    recs = U.read_records(FT.model_key, run_name)
    scores = U.score_run(val_examples, {i: recs[i]["query"] for i in val_ids}, val_ids)
    return {"val_EX": round(sum(s["ex"] for s in scores) / len(scores), 4),
            "val_EM": round(sum(s["em"] for s in scores) / len(scores), 4),
            "val_exec_errors": sum(s["exec_error"] for s in scores)}


def make_steps(epoch):
    """Shuffle (seeded per epoch), group by similar length, cut into micro-batches, then into optimizer steps."""
    rng = random.Random(FT.seed + epoch)
    ids = list(train_ids)
    rng.shuffle(ids)
    chunk = FT.micro_batch * 50  # sort within large chunks by length, so batches have similar lengths
    micro = []
    for c in range(0, len(ids), chunk):
        part = sorted(ids[c : c + chunk], key=lambda i: len(sequences[i]["input_ids"]))
        micro += [part[b : b + FT.micro_batch] for b in range(0, len(part), FT.micro_batch)]
    rng.shuffle(micro)  # random order of micro-batches
    return [micro[s : s + FT.grad_accum] for s in range(0, len(micro), FT.grad_accum)]


def pad(batch):
    """Right-pad a list of sequences into tensors (padding is masked out and never learned)."""
    width = max(len(sequences[i]["input_ids"]) for i in batch)
    ids = torch.full((len(batch), width), tokenizer.pad_token_id)
    labels = torch.full((len(batch), width), -100)
    mask = torch.zeros((len(batch), width), dtype=torch.long)
    for r, i in enumerate(batch):
        n = len(sequences[i]["input_ids"])
        ids[r, :n] = torch.tensor(sequences[i]["input_ids"])
        labels[r, :n] = torch.tensor(sequences[i]["labels"])
        mask[r, :n] = 1
    return ids.cuda(), labels.cuda(), mask.cuda()


def backward_on(model, batch, n_target):
    """Add the gradients of (sum of target-token losses of a micro-batch) / n_target; return that loss value.
    Long sequences are processed in smaller pieces; the result is the same, only memory use is lower."""
    width = max(len(sequences[i]["input_ids"]) for i in batch)
    piece = max(1, min(len(batch), 40_000 // width))  # at most ~40k tokens per forward pass (memory)
    total = 0.0
    for p in range(0, len(batch), piece):
        ids, labels, mask = pad(batch[p : p + piece])
        with torch.autocast("cuda", dtype=torch.bfloat16):  # mixed precision: compute in bf16, weights stay fp32
            hidden = model.model(input_ids=ids, attention_mask=mask).last_hidden_state
        # token t predicts token t+1; keep only positions whose next token is part of the target SQL
        next_tokens = labels[:, 1:]
        keep = next_tokens != -100
        logits = model.lm_head(hidden[:, :-1][keep]).float()  # vocabulary scores only where they are needed
        loss = F.cross_entropy(logits, next_tokens[keep], reduction="sum") / n_target
        loss.backward()  # gradients add up over pieces and micro-batches (gradient accumulation)
        total += loss.item()
    return total


def write_progress(**fields):
    """Record how far training is (phase, epoch, step, loss, validation results) for live monitoring."""
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    PROGRESS_FILE.write_text(json.dumps({**fields, "time": time.time()}))


def save_checkpoint(model, path):
    """Save the weights in bf16 (the precision used for evaluation) together with the tokenizer."""
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path, state_dict={k: v.to(torch.bfloat16) for k, v in model.state_dict().items()})
    tokenizer.save_pretrained(path)
    cfg = json.loads((path / "config.json").read_text())
    cfg["dtype"] = cfg["torch_dtype"] = "bfloat16"
    (path / "config.json").write_text(json.dumps(cfg, indent=2))


if BEST_DIR.exists() and SUMMARY_FILE.exists():
    summary = json.loads(SUMMARY_FILE.read_text())
    print(f"a fine-tuned model already exists ({BEST_DIR}); skipping training. Best epoch: {summary['best_epoch']}")
else:
    C.set_seed(FT.seed)
    validation = [{"epoch": 0, "train_loss": None, **validation_ex("ft_val_epoch0", None)}]  # the original model
    print("epoch 0 (original model):", validation[-1])

    model = transformers.AutoModelForCausalLM.from_pretrained(
        spec.hf_id, revision=spec.revision, dtype=torch.float32,  # fp32 master weights
        attn_implementation=spec.attn_implementation, token=C.get_hf_token()).cuda()
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    optimizer = torch.optim.AdamW(model.parameters(), lr=FT.learning_rate, weight_decay=FT.weight_decay)
    steps_per_epoch = len(make_steps(1))
    total_steps = steps_per_epoch * FT.epochs
    scheduler = transformers.get_cosine_schedule_with_warmup(optimizer, math.ceil(FT.warmup_ratio * total_steps), total_steps)
    print(f"{len(train_ids)} examples -> {steps_per_epoch} optimizer steps per epoch, {total_steps} in total")

    log, step, t_start = [], 0, time.time()
    progress = {"steps_per_epoch": steps_per_epoch, "total_steps": total_steps, "epochs": FT.epochs, "t_start": t_start}
    for epoch in range(1, FT.epochs + 1):
        model.train()
        t0, epoch_losses = time.time(), []
        for micro_batches in make_steps(epoch):
            # the loss of one optimizer step = mean over ALL target tokens of its micro-batches
            n_target = sum(sum(t != -100 for t in sequences[i]["labels"][1:]) for mb in micro_batches for i in mb)
            step_loss = sum(backward_on(model, mb, n_target) for mb in micro_batches)
            torch.nn.utils.clip_grad_norm_(model.parameters(), FT.max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            epoch_losses.append(step_loss)
            log.append({"epoch": epoch, "step": step, "lr": scheduler.get_last_lr()[0], "loss": round(step_loss, 5)})
            recent = [r["loss"] for r in log[-20:]]
            write_progress(**progress, phase="train", epoch=epoch, step=step, loss=sum(recent) / len(recent),
                           validation=validation)
            if step % 25 == 0:
                print(f"  epoch {epoch} step {step}/{total_steps}: loss {step_loss:.4f}, lr {scheduler.get_last_lr()[0]:.2e}")
        minutes = (time.time() - t0) / 60
        ckpt = RUN_DIR / f"epoch{epoch}"
        save_checkpoint(model, ckpt)
        model.eval()
        write_progress(**progress, phase="validate", epoch=epoch, step=step, validation=validation)
        validation.append({"epoch": epoch, "train_loss": round(sum(epoch_losses) / len(epoch_losses), 5),
                           "train_minutes": round(minutes, 1), **validation_ex(f"ft_val_epoch{epoch}", ckpt)})
        print(f"epoch {epoch} done in {minutes:.1f} min:", validation[-1])

    write_progress(**progress, phase="done", epoch=FT.epochs, step=step, validation=validation)
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()

    # keep the epoch with the highest validation EX (ties -> the earlier epoch); delete the other checkpoints
    best = max((v for v in validation if v["epoch"] > 0), key=lambda v: (v["val_EX"], -v["epoch"]))
    if BEST_DIR.exists():
        shutil.rmtree(BEST_DIR)
    (RUN_DIR / f"epoch{best['epoch']}").rename(BEST_DIR)
    for e in range(1, FT.epochs + 1):
        shutil.rmtree(RUN_DIR / f"epoch{e}", ignore_errors=True)

    pd.DataFrame(log).to_csv(C.TASK3_METRICS_DIR / "task3_training_log.csv", index=False)
    pd.DataFrame(validation).to_csv(C.TASK3_METRICS_DIR / "task3_validation.csv", index=False)
    summary = {"settings": {k: v for k, v in vars(FT).items()}, "best_epoch": best["epoch"], "validation": validation,
               "train_examples": len(train_ids), "val_examples": len(val_ids), "optimizer_steps": total_steps,
               "runtime": C.runtime_info(), "checkpoint": str(BEST_DIR.relative_to(C.REPO_ROOT))}
    SUMMARY_FILE.write_text(json.dumps(summary, indent=1) + "\n")
    print(f"best epoch: {best['epoch']} (validation EX {best['val_EX']}) -> {BEST_DIR}")

display(pd.read_csv(C.TASK3_METRICS_DIR / "task3_validation.csv"))

## Step 3.1 – Evaluate the fine-tuned model on the dev set (same as Task 1)

This cell runs the chosen fine-tuned model on **all 1034 dev questions** with exactly the Task 1 setup: the same prompts (`src/spider_dev_inputs.json`), greedy decoding, the same output limit, batch size 16, bf16 weights, eager attention, and the same SQL extraction. Only the weights differ. The outputs are saved as the run `finetuned` (`outputs/gemma3-1b/finetuned/responses.jsonl`), next to the original model's run `baseline`. Like Task 1, the run is resumable.

In [ ]:
dev_prompts = [ex["prompt"] for ex in dev_inputs]
ALL_IDS = list(range(len(dev_inputs)))
U.run_model(FT.model_key, ALL_IDS, "finetuned", dev_prompts, model_path=BEST_DIR)
display(pd.DataFrame([U.summarize(FT.model_key, run, n_full=len(ALL_IDS)) for run in ("baseline", "finetuned")],
                     index=["baseline (Task 1)", "fine-tuned"]))

## Step 3.2 – Score the fine-tuned model (same official scorer as Task 1)

This cell scores every fine-tuned prediction with the same official Spider evaluation code as Task 1, for **execution accuracy (EX)**, **exact match (EM)**, **EM (normalized)** and **component F1**. It saves the per-example scores to `outputs/task3/metrics/task3_per_example.csv`. As in Task 1, it then runs the official `evaluation.py` script itself and checks that its numbers equal ours at every difficulty level. The original model's scores are taken unchanged from Task 1 (`outputs/task1/metrics/task1_per_example.csv`).

In [ ]:
inputs = {ex["id"]: ex for ex in dev_inputs}
ft_recs = U.read_records(FT.model_key, "finetuned")
assert set(ft_recs) == set(ALL_IDS), "fine-tuned run is incomplete; rerun Step 3.1"
ft_scores = U.score_run(inputs, {i: ft_recs[i]["query"] for i in ALL_IDS}, ALL_IDS)
ft = pd.DataFrame([{"id": i, "difficulty": inputs[i]["difficulty"], "valid": ft_recs[i]["valid"], **s,
                    "output_tokens": ft_recs[i]["output_tokens"], "latency_s": ft_recs[i]["latency_s"]}
                   for i, s in zip(ALL_IDS, ft_scores)])
ft.to_csv(C.TASK3_METRICS_DIR / "task3_per_example.csv", index=False)

LEVELS_ALL = LEVELS + ["all"]
off = U.official_scores(inputs, {i: ft_recs[i]["query"] for i in ALL_IDS}, ALL_IDS)  # official script, run as a script
for k, level in enumerate(LEVELS_ALL):
    d = U.level_slice(ft, level)
    assert off["count"][k] == len(d)
    assert abs(off["execution"][k] - d["ex"].mean()) < 6e-4 and abs(off["exact match"][k] - d["em"].mean()) < 6e-4, level
    for c in U.COMPONENTS:
        assert abs(off[c][k] - U.component_scores(d, c)[2]) < 6e-4, (level, c)
print("fine-tuned model: EX, EM and component F1 match the official script at every difficulty level")

base = pd.read_csv(C.TASK1_METRICS_DIR / "task1_per_example.csv")
base = base[base["model"] == spec.name].set_index("id").loc[ALL_IDS].reset_index()  # Task 1 baseline, same ids

## Step 3.3 – Original vs. fine-tuned: comparison table and examples

**The main comparison table** (`task3_comparison.csv`) puts the original model (Task 1) and the fine-tuned model side by side. For EX, EM and EM (normalized) it gives the score at each difficulty level and overall, plus the change in percentage points. It also gives the number of invalid outputs, execution errors, output length and time per example.

**Question by question** (`task3_fixed_broken.csv`): because both models answered the same questions, we count, per difficulty level (EX):
- questions both models get right;
- questions **fixed** by fine-tuning (wrong before, right after);
- questions **broken** by fine-tuning (right before, wrong after);
- questions both get wrong.

**Examples** (`task3_examples.json`): to show what changed, the cell picks two fixed and two broken questions by a fixed rule. For fixed questions, the first one (lowest id) where the original query did not even run, and the first fixed hard or extra-hard question. For broken questions, the first broken medium question and the first broken hard or extra-hard question. Picking by rule rather than by hand avoids cherry-picking.

In [ ]:
def pct(x):
    return round(100 * float(x), 1)


rows = []
for metric, col in [("EX", "ex"), ("EM", "em"), ("EM (normalized)", "em_normalized")]:
    for level in LEVELS_ALL:
        b, f = U.level_slice(base, level)[col].mean(), U.level_slice(ft, level)[col].mean()
        rows.append({"metric": metric, "difficulty": level, "n": len(U.level_slice(ft, level)),
                     "original": round(b, 4), "fine_tuned": round(f, 4), "change_points": round(100 * (f - b), 1)})
for name, fn in [("no SQL extracted", lambda d: int((~d["valid"].astype(bool)).sum())),
                 ("execution errors", lambda d: int(d["exec_error"].astype(bool).sum())),
                 ("mean output tokens", lambda d: round(d["output_tokens"].mean(), 1)),
                 ("seconds per example", lambda d: round(d["latency_s"].mean(), 3))]:
    rows.append({"metric": name, "difficulty": "all", "n": len(ft), "original": fn(base), "fine_tuned": fn(ft),
                 "change_points": None})
comparison = pd.DataFrame(rows)
comparison.to_csv(C.TASK3_METRICS_DIR / "task3_comparison.csv", index=False)
display(comparison)

# question by question (EX): fixed = wrong before, right after; broken = right before, wrong after
both = base[["id", "difficulty", "ex"]].merge(ft[["id", "ex"]], on="id", suffixes=("_orig", "_ft"))
fb = []
for level in LEVELS_ALL:
    d = U.level_slice(both, level)
    o, f = d["ex_orig"].astype(bool), d["ex_ft"].astype(bool)
    fb.append({"difficulty": level, "n": len(d), "both_right": int((o & f).sum()), "fixed": int((~o & f).sum()),
               "broken": int((o & ~f).sum()), "both_wrong": int((~o & ~f).sum())})
fixed_broken = pd.DataFrame(fb)
fixed_broken.to_csv(C.TASK3_METRICS_DIR / "task3_fixed_broken.csv", index=False)
display(fixed_broken)

# examples, picked by a fixed rule (lowest id that meets each condition)
base_recs = U.read_records(FT.model_key, "baseline")
both = both.merge(base[["id", "exec_error"]], on="id")
rules = {"fixed: original query did not run": (both.ex_orig == 0) & (both.ex_ft == 1) & both.exec_error.astype(bool),
         "fixed: hard/extra question": (both.ex_orig == 0) & (both.ex_ft == 1) & both.difficulty.isin(["hard", "extra"]),
         "broken: medium question": (both.ex_orig == 1) & (both.ex_ft == 0) & (both.difficulty == "medium"),
         "broken: hard/extra question": (both.ex_orig == 1) & (both.ex_ft == 0) & both.difficulty.isin(["hard", "extra"])}
examples = []
for label, cond in rules.items():
    hits = both.loc[cond, "id"]
    if len(hits):
        i = int(hits.min())
        examples.append({"kind": label, "id": i, "difficulty": inputs[i]["difficulty"], "question": inputs[i]["question"],
                         "gold": inputs[i]["query"], "original": base_recs[i]["query"], "fine_tuned": ft_recs[i]["query"]})
(C.TASK3_METRICS_DIR / "task3_examples.json").write_text(json.dumps(examples, indent=1, ensure_ascii=False) + "\n")
for e in examples:
    print(f"\n[{e['kind']}] id {e['id']} ({e['difficulty']}): {e['question']}\n  gold:       {e['gold']}\n"
          f"  original:   {e['original']}\n  fine-tuned: {e['fine_tuned']}")

## Step 3.4 – Plots

This cell draws two figures and saves each as PNG and PDF in `outputs/task3/figures/`:
1. **`task3_comparison`:** EX and EM for each difficulty level, for the original model (light, hatched) and the fine-tuned model (solid), on the same 0–1 scale. EM (normalized) is not plotted because it equals EM for both models (neither writes `INNER JOIN`); its values are in `task3_comparison.csv`.
2. **`task3_training`:** how training went.
   - (a) the training loss over the optimizer steps, a moving average of how wrong the predictions on the training data are; it should go down;
   - (b) validation EX after each epoch, with epoch 0 being the original model. The chosen epoch is marked.

In [ ]:
import matplotlib.pyplot as plt

BLUE = "#2a78d6"  # Gemma 3 1B's color in all Task 1 figures
INK, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": AXIS, "axes.labelcolor": MUTED, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
                     "pdf.fonttype": 42, "hatch.color": BLUE})
C.TASK3_FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    fig.savefig(C.TASK3_FIGURES_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(C.TASK3_FIGURES_DIR / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None})
    plt.show()


# 1. original vs fine-tuned by difficulty
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
w = 0.38
for ax, metric in zip(axes, ["EX", "EM"]):  # EM (normalized) equals EM here: neither model writes INNER JOIN
    c = comparison[comparison["metric"] == metric].set_index("difficulty").loc[LEVELS_ALL]
    xs = range(len(LEVELS_ALL))
    b1 = ax.bar([x - w / 2 for x in xs], c["original"], w, color="white", edgecolor=BLUE, hatch="///", linewidth=1.5, label="original (Task 1)")
    b2 = ax.bar([x + w / 2 for x in xs], c["fine_tuned"], w, color=BLUE, edgecolor="white", linewidth=1.5, label="fine-tuned")
    for bars in (b1, b2):
        ax.bar_label(bars, labels=[f"{v:.2f}" for v in bars.datavalues], fontsize=7, color=MUTED, padding=2)
    ax.set_xticks(list(xs), [l if l != "all" else "overall" for l in LEVELS_ALL])
    ax.set_title(f"{metric}, Gemma 3 1B", color=INK, fontsize=11, loc="left")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
axes[0].set_ylabel("share of dev examples correct")
axes[0].legend(frameon=False, fontsize=9, loc="upper right")
fig.tight_layout()
save(fig, "task3_comparison")

# 2. training loss and validation EX
log_df = pd.read_csv(C.TASK3_METRICS_DIR / "task3_training_log.csv")
val_df = pd.read_csv(C.TASK3_METRICS_DIR / "task3_validation.csv")
best_epoch = json.loads(SUMMARY_FILE.read_text())["best_epoch"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(log_df["step"], log_df["loss"], color=GRID, linewidth=1)
ax1.plot(log_df["step"], log_df["loss"].rolling(20, min_periods=1).mean(), color=BLUE, linewidth=2)
ax1.set_xlabel("optimizer step")
ax1.set_ylabel("training loss (mean per target token)")
ax1.set_title("(a) Training loss (thin: per step, thick: moving average)", color=INK, fontsize=10, loc="left")
ax1.grid(axis="y", color=GRID, linewidth=0.8)
ax2.plot(val_df["epoch"], val_df["val_EX"], color=BLUE, marker="o", linewidth=2)
for _, r in val_df.iterrows():
    ax2.annotate(f"{r['val_EX']:.3f}", (r["epoch"], r["val_EX"]), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, color=MUTED)
ax2.scatter([best_epoch], [val_df.set_index("epoch").loc[best_epoch, "val_EX"]], s=160, facecolor="none", edgecolor=INK, zorder=3)
ax2.set_xticks(val_df["epoch"], ["0\n(original)"] + [str(e) for e in val_df["epoch"][1:]])
ax2.set_ylim(0, 1)
ax2.set_xlabel("epoch")
ax2.set_ylabel("validation EX")
ax2.set_title(f"(b) Validation EX after each epoch (circled: chosen epoch {best_epoch})", color=INK, fontsize=10, loc="left")
ax2.grid(axis="y", color=GRID, linewidth=0.8)
fig.tight_layout()
save(fig, "task3_training")